In [6]:
import pandas as pd

df = pd.read_csv("county-population-data-tidy.csv")
# FIPS codes get read as ints — restore the leading zero so they match the topojson ids
df['county'] = df['county'].astype(str).str.zfill(5)
team_list = sorted(df['team'].unique())
print(f"{len(team_list)} teams: {team_list}")

47 teams: ['ALG', 'ARG', 'AUS', 'AUT', 'BEL', 'BIH', 'BRA', 'CAN', 'CIV', 'COD', 'COL', 'CPV', 'CRO', 'CUW', 'CZE', 'ECU', 'EGY', 'ENG', 'ESP', 'FRA', 'GER', 'GHA', 'HAI', 'IRN', 'IRQ', 'JOR', 'JPN', 'KOR', 'KSA', 'MAR', 'MEX', 'NED', 'NOR', 'NZL', 'PAN', 'PAR', 'POR', 'QAT', 'RSA', 'SCO', 'SEN', 'SUI', 'SWE', 'TUN', 'TUR', 'URU', 'UZB']


In [7]:
import json
import topojson

# From https://unpkg.com/us-atlas@3/counties-10m.json
with open("counties-10m.json", "r") as f:
    map_data = json.load(f)

# Parse the topojson into GeoDataFrames for counties and states.
# The topojson `id` becomes the GeoDataFrame index (5-digit FIPS for counties, 2-digit for states).
counties_gdf = topojson.Topology(map_data, object_name="counties").to_gdf()
states_gdf = topojson.Topology(map_data, object_name="states").to_gdf()

counties_gdf = counties_gdf.set_crs("EPSG:4326")
states_gdf = states_gdf.set_crs("EPSG:4326")

counties_gdf["fips"] = counties_gdf.index.astype(str).str.zfill(5)
counties_gdf["state_fips"] = counties_gdf["fips"].str[:2]
states_gdf["state_fips"] = states_gdf.index.astype(str).str.zfill(2)

# Restrict to the continental US (drop AK, HI, and the territories)
non_continental = {"02", "15", "60", "66", "69", "72", "78"}
counties_gdf = counties_gdf[~counties_gdf["state_fips"].isin(non_continental)].copy()
states_gdf = states_gdf[~states_gdf["state_fips"].isin(non_continental)].copy()

# Reproject into Albers Equal Area Conic for a familiar continental US shape
counties_albers = counties_gdf.to_crs("EPSG:5070")
states_albers = states_gdf.to_crs("EPSG:5070")

print(f"counties: {len(counties_albers)}  states: {len(states_albers)}")

counties: 3108  states: 49


In [8]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Map bounds drive the figure aspect ratio so 800px wide always lands on the same continental US frame.
MINX, MINY, MAXX, MAXY = counties_albers.total_bounds
ASPECT = (MAXY - MINY) / (MAXX - MINX)

IMAGES_DIR = "images"
os.makedirs(IMAGES_DIR, exist_ok=True)

def render_team_map(team, width_px=800, dpi=100):
    """Render an 800px-wide choropleth for one team, mirroring d3.scaleSequentialSqrt+interpolateBlues."""
    team_pop = (
        df[df["team"] == team][["county", "pop"]]
        .rename(columns={"county": "fips"})
    )
    merged = counties_albers.merge(team_pop, on="fips", how="left")
    merged["pop"] = merged["pop"].fillna(0)

    # sqrt scale to match d3.scaleSequentialSqrt
    max_pop = merged["pop"].max()
    merged["pop_scaled"] = np.sqrt(merged["pop"])
    vmax = np.sqrt(max_pop) if max_pop > 0 else 1.0

    height_px = int(round(width_px * ASPECT))
    fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])

    merged.plot(
        ax=ax,
        column="pop_scaled",
        cmap="Blues",
        vmin=0,
        vmax=vmax,
        edgecolor="none",
        linewidth=0,
    )
    # State borders on top, drawn in white like the Observable example
    states_albers.boundary.plot(ax=ax, color="white", linewidth=0.5)

    ax.set_xlim(MINX, MAXX)
    ax.set_ylim(MINY, MAXY)
    ax.set_axis_off()

    output_path = os.path.join(IMAGES_DIR, f"{team}.png")
    fig.savefig(output_path, dpi=dpi)
    plt.close(fig)
    return output_path

In [9]:
for team in team_list:
    path = render_team_map(team)
    print(f"  saved {path}")

  saved images/ALG.png
  saved images/ARG.png
  saved images/AUS.png
  saved images/AUT.png
  saved images/BEL.png
  saved images/BIH.png
  saved images/BRA.png
  saved images/CAN.png
  saved images/CIV.png
  saved images/COD.png
  saved images/COL.png
  saved images/CPV.png
  saved images/CRO.png
  saved images/CUW.png
  saved images/CZE.png
  saved images/ECU.png
  saved images/EGY.png
  saved images/ENG.png
  saved images/ESP.png
  saved images/FRA.png
  saved images/GER.png
  saved images/GHA.png
  saved images/HAI.png
  saved images/IRN.png
  saved images/IRQ.png
  saved images/JOR.png
  saved images/JPN.png
  saved images/KOR.png
  saved images/KSA.png
  saved images/MAR.png
  saved images/MEX.png
  saved images/NED.png
  saved images/NOR.png
  saved images/NZL.png
  saved images/PAN.png
  saved images/PAR.png
  saved images/POR.png
  saved images/QAT.png
  saved images/RSA.png
  saved images/SCO.png
  saved images/SEN.png
  saved images/SUI.png
  saved images/SWE.png
  saved ima